# FIFA Analytics Dashboard - setup and usage notes

Notes and helper code for running the three Streamlit apps in this repo
(`app.py`, `advanced_analytics.py`, `tactical_analysis.py`) on a local machine.

Contents:

1. Python environment
2. Dependencies
3. Data validation and loading
4. Running the apps
5. Ports
6. Performance (dtypes, sampling, chunked reading)
7. Caching
8. Troubleshooting, testing and deployment notes

Requirements: Python 3.9+, the FIFA 23 player CSVs in `data/` (see `data/README.md`).
The helpers in sections 5 and 6 also need `psutil`.

## 1. Python environment

It is easiest to work in a virtual environment so the pinned package versions
do not clash with other projects. The cells below list the commands and check
whether the notebook kernel is running inside a venv.

In [ ]:
# Windows Environment Setup Commands
# Run these in Command Prompt or PowerShell

# 1. Navigate to your project directory
# cd path\to\fifa-analytics-dashboard

# 2. Create virtual environment
# python -m venv venv

# 3. Activate virtual environment (Windows Command Prompt)
# venv\Scripts\activate

# 4. Activate virtual environment (Windows PowerShell)
# venv\Scripts\Activate.ps1

# 5. Verify Python version and virtual environment
import sys
import os

def check_environment():
    """Check current Python environment details"""
    print("Python Environment Information")
    print("=" * 40)
    print(f"Python Version: {sys.version}")
    print(f"Python Executable: {sys.executable}")
    print(f"Virtual Environment: {'Yes' if hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix) else 'No'}")
    print(f"Current Working Directory: {os.getcwd()}")
    print(f"Python Path: {sys.path[0]}")
    
    # Check if running in virtual environment
    in_venv = (hasattr(sys, 'real_prefix') or 
               (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix))
    
    if in_venv:
        print("Running in virtual environment")
    else:
        print(" Running in global Python environment")
    
    return in_venv

# Execute the check
environment_status = check_environment()

In [ ]:
# macOS/Linux Environment Setup Commands
# Run these in Terminal

# 1. Navigate to your project directory
# cd ~/fifa-analytics-dashboard

# 2. Create virtual environment
# python3 -m venv venv

# 3. Activate virtual environment
# source venv/bin/activate

# 4. Alternative: Using conda (if Anaconda/Miniconda is installed)
# conda create -n fifa_dash python=3.12
# conda activate fifa_dash

# Cross-platform environment verification
def verify_python_setup():
    """Verify Python setup is suitable for the FIFA Dashboard"""
    print("Python Setup Verification")
    print("=" * 35)
    
    # Check Python version
    python_version = sys.version_info
    min_version = (3, 8)
    
    print(f"Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
    
    if python_version >= min_version:
        print("Python version is compatible")
    else:
        print(f"Python version too old. Minimum required: {min_version[0]}.{min_version[1]}")
        return False
    
    # Check available modules for dashboard
    required_modules = ['sys', 'os', 'subprocess']
    missing_modules = []
    
    for module in required_modules:
        try:
            __import__(module)
            print(f"{module} available")
        except ImportError:
            missing_modules.append(module)
            print(f"{module} missing")
    
    if not missing_modules:
        print("Python setup looks fine")
        return True
    else:
        print(f" Missing modules: {missing_modules}")
        return False

# Run verification
setup_ok = verify_python_setup()

## 2. Dependencies

Packages used by the apps:

- `streamlit` - web UI
- `pandas`, `numpy` - data handling
- `plotly` - interactive charts
- `scikit-learn` - value prediction (random forest / gradient boosting) and K-means clustering
- `matplotlib` - only used by `demo.py` for a test plot

Install them with `pip install -r requirements.txt`. The cell below checks which ones are installed.

In [ ]:
# Installation Commands (run in terminal)
# pip install -r requirements.txt
# OR install packages individually:
# pip install streamlit pandas numpy plotly scikit-learn

import subprocess
import sys
from importlib import metadata
from typing import List, Dict

def check_package_installation(packages: List[str]) -> Dict[str, bool]:
    """Check if required packages are installed"""
    results = {}
    
    print("Checking Package Installation Status")
    print("=" * 45)
    
    for package in packages:
        try:
            # Try to get package information
            version = metadata.version(package)
            results[package] = True
            print(f"{package}: {version}")
        except metadata.PackageNotFoundError:
            results[package] = False
            print(f"{package}: not installed")
        except Exception as e:
            results[package] = False
            print(f"{package}: error while checking - {e}")
    
    return results

def install_missing_packages(missing_packages: List[str]):
    """Install missing packages using pip"""
    if not missing_packages:
        print("All packages are already installed.")
        return
    
    print(f"\nInstalling missing packages: {missing_packages}")
    
    for package in missing_packages:
        try:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"{package} installed successfully")
        except subprocess.CalledProcessError as e:
            print(f"Failed to install {package}: {e}")

# Define required packages for FIFA Dashboard
required_packages = [
    "streamlit",
    "pandas", 
    "numpy",
    "plotly",
    "scikit-learn",
    "matplotlib"
]

# Check installation status
installation_status = check_package_installation(required_packages)

# Find missing packages
missing_packages = [pkg for pkg, installed in installation_status.items() if not installed]

print(f"\nInstallation Summary:")
print(f"   Total packages: {len(required_packages)}")
print(f"   Installed: {len(required_packages) - len(missing_packages)}")
print(f"   Missing: {len(missing_packages)}")

if missing_packages:
    print(f"\n Missing packages: {missing_packages}")
    print("Run the following command to install them:")
    print(f"pip install {' '.join(missing_packages)}")
else:
    print("\nAll required packages are installed.")

## 3. Data validation and loading

The apps read `data/male_players.csv` and `data/female_players.csv` from the
[FIFA 23 Complete Player Dataset](https://www.kaggle.com/datasets/stefanoleone992/fifa-23-complete-player-dataset).

Columns the code relies on:

- `player_id`, `long_name`
- `overall`, `potential`, `value_eur`, `age`
- `pace`, `shooting`, `passing`, `dribbling`, `defending`, `physic`
- `club_name`, `player_positions`, `nationality_name`, `league_name`

The functions below check that the files exist, contain these columns and look sane,
then load and combine them.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

def validate_data_structure(file_path: str) -> Dict:
    """Validate FIFA data file structure and quality"""
    
    validation_results = {
        'file_exists': False,
        'file_readable': False,
        'required_columns_present': False,
        'data_quality': {},
        'recommendations': []
    }
    
    print(f"Validating: {file_path}")
    print("=" * 50)
    
    # Check if file exists
    if not Path(file_path).exists():
        print(f"File not found: {file_path}")
        validation_results['recommendations'].append(f"Create or move {file_path} to the correct location")
        return validation_results
    
    validation_results['file_exists'] = True
    print(f"File exists: {file_path}")
    
    try:
        # Try to read the file
        df = pd.read_csv(file_path, low_memory=False, nrows=1000)  # Read first 1000 rows for validation
        validation_results['file_readable'] = True
        print(f"File readable: {len(df)} rows sampled")
        
        # Check required columns
        required_columns = [
            'player_id', 'long_name', 'overall', 'potential', 
            'value_eur', 'age', 'pace', 'shooting', 'passing', 
            'dribbling', 'defending', 'physic'
        ]
        
        missing_columns = [col for col in required_columns if col not in df.columns]
        
        if not missing_columns:
            validation_results['required_columns_present'] = True
            print("All required columns present")
        else:
            print(f"Missing columns: {missing_columns}")
            validation_results['recommendations'].append(f"Add missing columns: {missing_columns}")
        
        # Data quality checks
        quality_checks = {}
        
        # Check for null values in critical columns
        for col in ['player_id', 'long_name', 'overall']:
            if col in df.columns:
                null_count = df[col].isnull().sum()
                null_percentage = (null_count / len(df)) * 100
                quality_checks[f'{col}_null_percentage'] = null_percentage
                
                if null_percentage == 0:
                    print(f"{col}: No null values")
                elif null_percentage < 5:
                    print(f" {col}: {null_percentage:.1f}% null values (acceptable)")
                else:
                    print(f"{col}: {null_percentage:.1f}% null values (problematic)")
                    validation_results['recommendations'].append(f"Clean null values in {col}")
        
        # Check data types
        if 'overall' in df.columns:
            if df['overall'].dtype in ['int64', 'float64']:
                overall_range = (df['overall'].min(), df['overall'].max())
                quality_checks['overall_range'] = overall_range
                
                if 0 <= overall_range[0] and overall_range[1] <= 100:
                    print(f"Overall ratings in valid range: {overall_range}")
                else:
                    print(f" Overall ratings outside expected range: {overall_range}")
                    validation_results['recommendations'].append("Check overall rating values")
        
        # Check for duplicates
        if 'player_id' in df.columns:
            duplicate_count = df['player_id'].duplicated().sum()
            quality_checks['duplicate_players'] = duplicate_count
            
            if duplicate_count == 0:
                print("No duplicate player IDs")
            else:
                print(f" {duplicate_count} duplicate player IDs found")
                validation_results['recommendations'].append("Remove duplicate player records")
        
        validation_results['data_quality'] = quality_checks
        
        # File size info
        file_size_mb = Path(file_path).stat().st_size / (1024 * 1024)
        print(f"File size: {file_size_mb:.1f} MB")
        
        if file_size_mb > 100:
            validation_results['recommendations'].append("Consider data sampling for better performance")
        
    except Exception as e:
        print(f"Error reading file: {str(e)}")
        validation_results['recommendations'].append(f"Fix file format or encoding issues: {str(e)}")
    
    return validation_results

def load_fifa_data(data_path: str = "data/") -> Optional[pd.DataFrame]:
    """Load and combine FIFA player data"""
    
    print("Loading FIFA Player Data")
    print("=" * 30)
    
    male_file = Path(data_path) / "male_players.csv"
    female_file = Path(data_path) / "female_players.csv"
    
    # Validate files
    male_validation = validate_data_structure(str(male_file))
    female_validation = validate_data_structure(str(female_file))
    
    print("\n" + "="*50)
    
    # Load data if validation passes
    dataframes = []
    
    if male_validation['file_readable'] and male_validation['required_columns_present']:
        try:
            male_df = pd.read_csv(male_file, low_memory=False)
            male_df['gender'] = 'Male'
            dataframes.append(male_df)
            print(f"Loaded {len(male_df):,} male players")
        except Exception as e:
            print(f"Error loading male players: {e}")
    
    if female_validation['file_readable'] and female_validation['required_columns_present']:
        try:
            female_df = pd.read_csv(female_file, low_memory=False)
            female_df['gender'] = 'Female'
            dataframes.append(female_df)
            print(f"Loaded {len(female_df):,} female players")
        except Exception as e:
            print(f"Error loading female players: {e}")
    
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        print(f"Total players loaded: {len(combined_df):,}")
        
        # Basic data info
        print(f"\nDataset Overview:")
        print(f"   Shape: {combined_df.shape}")
        print(f"   Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        print(f"   Age range: {combined_df['age'].min()}-{combined_df['age'].max()}")
        print(f"   Overall range: {combined_df['overall'].min()}-{combined_df['overall'].max()}")
        
        return combined_df
    else:
        print("No valid data files found")
        return None

# Execute data validation and loading
fifa_data = load_fifa_data()

## 4. Running the apps

There are three separate Streamlit apps:

1. `app.py` - overview, single player view, two-player comparison, clustering and value prediction
2. `advanced_analytics.py` - player report with similar players, market analysis, value model with feature importance, league comparison
3. `tactical_analysis.py` - K-means clustering with PCA plot, best players per formation, 11-player team builder

Each one is started with `streamlit run <file>`. The helper below prints the commands and checks that the files exist.

In [ ]:
import subprocess
import time
import webbrowser
from pathlib import Path

class StreamlitManager:
    """Manage multiple Streamlit applications"""
    
    def __init__(self, project_path: str = "."):
        self.project_path = Path(project_path)
        self.processes = {}
        self.apps = {
            "main": {
                "file": "app.py",
                "port": 8501,
                "name": "Main Dashboard",
                "description": "Core analytics and player comparison"
            },
            "advanced": {
                "file": "advanced_analytics.py", 
                "port": 8502,
                "name": "Advanced Analytics",
                "description": "ML predictions and market analysis"
            },
            "tactical": {
                "file": "tactical_analysis.py",
                "port": 8503,
                "name": "Tactical Analysis", 
                "description": "Team building and formations"
            }
        }
    
    def show_commands(self):
        """Display all available Streamlit commands"""
        print("FIFA Dashboard - Streamlit Commands")
        print("=" * 45)
        
        print("\nIndividual Application Commands:")
        print("-" * 35)
        
        for app_key, app_info in self.apps.items():
            print(f"\n{app_info['name']}:")
            print(f"   File: {app_info['file']}")
            print(f"   Port: {app_info['port']}")
            print(f"   Description: {app_info['description']}")
            print(f"   Command: streamlit run {app_info['file']} --server.port {app_info['port']}")
            print(f"   URL: http://localhost:{app_info['port']}")
        
        print(f"\nAdvanced Commands:")
        print("-" * 20)
        print("Custom theme:")
        print("   streamlit run app.py --theme.primaryColor '#1f77b4'")
        
        print("\nNetwork access:")
        print("   streamlit run app.py --server.address 0.0.0.0")
        
        print("\nDebug mode:")
        print("   streamlit run app.py --logger.level debug")
        
        print("\nPerformance:")
        print("   streamlit run app.py --server.maxUploadSize 1024")
        
        print("\nStop all applications:")
        print("   Use Ctrl+C in each terminal or close terminal windows")
    
    def check_file_exists(self, filename: str) -> bool:
        """Check if Streamlit app file exists"""
        file_path = self.project_path / filename
        exists = file_path.exists()
        
        if exists:
            print(f"Found: {filename}")
        else:
            print(f"Missing: {filename}")
            print(f"   Expected path: {file_path}")
        
        return exists
    
    def validate_apps(self):
        """Validate all application files exist"""
        print("Validating Application Files")
        print("=" * 35)
        
        all_valid = True
        for app_key, app_info in self.apps.items():
            valid = self.check_file_exists(app_info['file'])
            if not valid:
                all_valid = False
        
        if all_valid:
            print("\nAll application files found!")
            print("Ready to launch dashboards!")
        else:
            print("\nSome application files are missing.")
            print("Please ensure you're in the correct directory.")
        
        return all_valid
    
    def get_launch_instructions(self):
        """Get step-by-step launch instructions"""
        instructions = [
            "Step-by-Step Launch Instructions",
            "=" * 40,
            "",
            "1. Open three separate terminal/command prompt windows",
            "",
            "2. Navigate to project directory in each:",
            f'   cd "{self.project_path}"',
            "",
            "3. Activate virtual environment in each:",
            "   Windows: venv\\Scripts\\activate",
            "   macOS/Linux: source venv/bin/activate",
            "",
            "4. Run applications in separate terminals:",
            ""
        ]
        
        for i, (app_key, app_info) in enumerate(self.apps.items(), 1):
            instructions.extend([
                f"   Terminal {i} - {app_info['name']}:",
                f"   streamlit run {app_info['file']} --server.port {app_info['port']}",
                ""
            ])
        
        instructions.extend([
            "5. Access dashboards in browser:",
            ""
        ])
        
        for app_key, app_info in self.apps.items():
            instructions.append(f"   {app_info['name']}: http://localhost:{app_info['port']}")
        
        instructions.extend([
            "",
            "Notes:",
            "   - Use Ctrl+C to stop an application",
            "   - Applications auto-reload on code changes",
            "   - Check browser console for any errors",
            "   - Use different browsers for multiple apps"
        ])
        
        return "\n".join(instructions)

# Initialize Streamlit Manager
streamlit_manager = StreamlitManager()

# Show all available commands
streamlit_manager.show_commands()

print("\n" + "="*60)

# Validate applications
apps_valid = streamlit_manager.validate_apps()

print("\n" + "="*60)

# Show launch instructions
if apps_valid:
    print(streamlit_manager.get_launch_instructions())
else:
    print(" Fix missing files before launching applications")

## 5. Ports

Streamlit uses port 8501 by default and moves to the next free port if it is taken.
To run all three apps at the same time with stable URLs, give each one its own port:

- `app.py` - 8501
- `advanced_analytics.py` - 8502
- `tactical_analysis.py` - 8503

The cell below checks which of these ports are free and writes small launcher
scripts (`run_*.bat`, `launch_dashboard.ps1`) for Windows. These files are ignored by git.

In [ ]:
import socket
import psutil
from typing import List, Dict

class PortManager:
    """Manage ports for multiple Streamlit applications"""
    
    def __init__(self):
        self.default_ports = {
            'main_dashboard': 8501,
            'advanced_analytics': 8502, 
            'tactical_analysis': 8503
        }
        self.port_range = range(8501, 8520)  # Check ports 8501-8519
    
    def check_port_availability(self, port: int) -> bool:
        """Check if a port is available"""
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                sock.settimeout(1)
                result = sock.connect_ex(('localhost', port))
                return result != 0  # Port is available if connection fails
        except Exception:
            return False
    
    def find_process_on_port(self, port: int) -> Dict:
        """Find which process is using a specific port"""
        for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
            try:
                conns = proc.net_connections() if hasattr(proc, 'net_connections') else proc.connections()
                for conn in conns:
                    if conn.laddr.port == port:
                        return {
                            'pid': proc.info['pid'],
                            'name': proc.info['name'],
                            'cmdline': ' '.join(proc.info['cmdline']) if proc.info['cmdline'] else 'N/A'
                        }
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
        return None
    
    def scan_ports(self) -> Dict:
        """Scan all relevant ports and return status"""
        print("Port Availability Scan")
        print("=" * 30)
        
        port_status = {}
        
        for app_name, port in self.default_ports.items():
            available = self.check_port_availability(port)
            port_status[port] = {
                'app': app_name,
                'available': available,
                'process': None if available else self.find_process_on_port(port)
            }
            
            print(f"Port {port} ({app_name}): {'Available' if available else 'In Use'}")
            
            if not available:
                process_info = port_status[port]['process']
                if process_info:
                    print(f"   Process: {process_info['name']} (PID: {process_info['pid']})")
                    if 'streamlit' in process_info['cmdline'].lower():
                        print(f"   Streamlit app detected")
        
        return port_status
    
    def find_available_ports(self, count: int = 3) -> List[int]:
        """Find available ports for applications"""
        available_ports = []
        
        for port in self.port_range:
            if self.check_port_availability(port):
                available_ports.append(port)
                if len(available_ports) >= count:
                    break
        
        return available_ports
    
    def generate_port_commands(self) -> Dict[str, str]:
        """Generate Streamlit commands with appropriate ports"""
        port_status = self.scan_ports()
        commands = {}
        
        print(f"\nRecommended Commands:")
        print("-" * 25)
        
        # Try to use default ports first, fallback to available ones
        available_ports = self.find_available_ports()
        port_assignments = {}
        
        for i, (app_name, default_port) in enumerate(self.default_ports.items()):
            if port_status[default_port]['available']:
                assigned_port = default_port
            else:
                assigned_port = available_ports[i] if i < len(available_ports) else default_port + 10
            
            port_assignments[app_name] = assigned_port
        
        # Generate commands
        app_files = {
            'main_dashboard': 'app.py',
            'advanced_analytics': 'advanced_analytics.py',
            'tactical_analysis': 'tactical_analysis.py'
        }
        
        for app_name, port in port_assignments.items():
            file_name = app_files[app_name]
            command = f"streamlit run {file_name} --server.port {port}"
            commands[app_name] = {
                'command': command,
                'port': port,
                'url': f"http://localhost:{port}"
            }
            
            print(f"\n{app_name.replace('_', ' ').title()}:")
            print(f"   Command: {command}")
            print(f"   URL: http://localhost:{port}")
        
        return commands
    
    def create_batch_files(self, commands: Dict) -> None:
        """Create batch files for easy application launching"""
        print(f"\nCreating Launch Scripts:")
        print("-" * 30)
        
        # Windows batch files
        for app_name, cmd_info in commands.items():
            batch_content = f"""@echo off
echo Starting {app_name.replace('_', ' ').title()}...
echo URL: {cmd_info['url']}
echo Press Ctrl+C to stop the application
echo.
{cmd_info['command']}
pause"""
            
            batch_file = f"run_{app_name}.bat"
            try:
                with open(batch_file, 'w') as f:
                    f.write(batch_content)
                print(f"Created: {batch_file}")
            except Exception as e:
                print(f"Failed to create {batch_file}: {e}")
        
        # PowerShell script for all apps
        ps_content = """# FIFA Dashboard Launcher
Write-Host "FIFA Analytics Dashboard Launcher" -ForegroundColor Green
Write-Host "=" * 45 -ForegroundColor Green

$apps = @{"""
        
        for app_name, cmd_info in commands.items():
            ps_content += f"""
    "{app_name}" = @{{
        "command" = "{cmd_info['command']}"
        "url" = "{cmd_info['url']}"
    }}"""
        
        ps_content += """
}

Write-Host "Choose an application to launch:" -ForegroundColor Yellow
$i = 1
foreach ($app in $apps.Keys) {
    Write-Host "$i. $($app.Replace('_', ' '))" -ForegroundColor Cyan
    $i++
}

$choice = Read-Host "Enter your choice (1-3)"
$appNames = @($apps.Keys)
$selectedApp = $appNames[$choice - 1]

if ($selectedApp) {
    $appInfo = $apps[$selectedApp]
    Write-Host "Launching $($selectedApp.Replace('_', ' '))..." -ForegroundColor Green
    Write-Host "URL: $($appInfo.url)" -ForegroundColor Yellow
    Write-Host " Press Ctrl+C to stop" -ForegroundColor Red
    Write-Host ""
    
    # Execute the command
    Invoke-Expression $appInfo.command
} else {
    Write-Host "Invalid choice" -ForegroundColor Red
}"""
        
        try:
            with open("launch_dashboard.ps1", 'w') as f:
                f.write(ps_content)
            print("Created: launch_dashboard.ps1")
        except Exception as e:
            print(f"Failed to create PowerShell script: {e}")

# Initialize Port Manager
port_manager = PortManager()

# Scan current port status
current_status = port_manager.scan_ports()

print("\n" + "="*50)

# Generate optimized commands
optimized_commands = port_manager.generate_port_commands()

print("\n" + "="*50)

# Create convenience scripts
port_manager.create_batch_files(optimized_commands)

print("\nNotes:")
print("- Use the generated batch files for easy launching")
print("- Run each app in a separate terminal window")
print("- Keep terminals open while using the dashboards")
print("- Use Ctrl+C to gracefully stop applications")

## 6. Performance

The player files can be large, so a few things help when working with them:

- downcasting numeric columns and converting low-cardinality strings to `category`
- working on a sample while developing
- reading the CSV in chunks when only aggregates are needed

The cell below tries these on a mock dataset so it runs without the real data.

In [ ]:
import pandas as pd
import numpy as np
import time
from typing import Dict, List, Optional, Tuple
import psutil
import gc

class PerformanceOptimizer:
    """Tools for optimizing FIFA dashboard performance"""
    
    def __init__(self):
        self.memory_usage_history = []
        
    def get_memory_usage(self) -> float:
        """Get current memory usage in MB"""
        process = psutil.Process()
        return process.memory_info().rss / 1024 / 1024
    
    def optimize_dataframe_dtypes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Optimize DataFrame data types to reduce memory usage"""
        print("Optimizing DataFrame Data Types")
        print("=" * 35)
        
        initial_memory = df.memory_usage(deep=True).sum() / 1024**2
        print(f"Initial memory usage: {initial_memory:.1f} MB")
        
        optimized_df = df.copy()
        
        # Optimize integer columns
        int_columns = df.select_dtypes(include=['int64']).columns
        for col in int_columns:
            max_val = df[col].max()
            min_val = df[col].min()
            
            if min_val >= 0:  # Unsigned integers
                if max_val < 255:
                    optimized_df[col] = df[col].astype('uint8')
                elif max_val < 65535:
                    optimized_df[col] = df[col].astype('uint16')
                elif max_val < 4294967295:
                    optimized_df[col] = df[col].astype('uint32')
            else:  # Signed integers
                if min_val > -128 and max_val < 127:
                    optimized_df[col] = df[col].astype('int8')
                elif min_val > -32768 and max_val < 32767:
                    optimized_df[col] = df[col].astype('int16')
                elif min_val > -2147483648 and max_val < 2147483647:
                    optimized_df[col] = df[col].astype('int32')
        
        # Optimize float columns
        float_columns = df.select_dtypes(include=['float64']).columns
        for col in float_columns:
            optimized_df[col] = pd.to_numeric(df[col], downcast='float')
        
        # Convert categorical columns
        object_columns = df.select_dtypes(include=['object']).columns
        categorical_threshold = 0.5  # Convert to category if unique ratio < 50%
        
        for col in object_columns:
            unique_ratio = df[col].nunique() / len(df)
            if unique_ratio < categorical_threshold:
                optimized_df[col] = df[col].astype('category')
                print(f"Converted {col} to category (unique ratio: {unique_ratio:.2%})")
        
        final_memory = optimized_df.memory_usage(deep=True).sum() / 1024**2
        memory_reduction = (initial_memory - final_memory) / initial_memory * 100
        
        print(f"Final memory usage: {final_memory:.1f} MB")
        print(f"Memory reduction: {memory_reduction:.1f}%")
        
        return optimized_df
    
    def create_sample_dataset(self, df: pd.DataFrame, sample_size: int = 10000, 
                            strategy: str = 'random') -> pd.DataFrame:
        """Create a representative sample of the dataset"""
        print(f"Creating Sample Dataset ({strategy})")
        print("=" * 35)
        
        if len(df) <= sample_size:
            print(f"Dataset already small enough: {len(df)} rows")
            return df
        
        if strategy == 'random':
            sample_df = df.sample(n=sample_size, random_state=42)
            print(f"Random sampling: {len(sample_df)} rows")
            
        elif strategy == 'stratified':
            # Stratified sampling by overall rating
            if 'overall' in df.columns:
                df['overall_bin'] = pd.cut(df['overall'], bins=10, labels=False)
                sample_df = df.groupby('overall_bin').apply(
                    lambda x: x.sample(min(len(x), sample_size // 10), random_state=42)
                ).reset_index(drop=True)
                sample_df = sample_df.drop('overall_bin', axis=1)
                print(f"Stratified sampling: {len(sample_df)} rows")
            else:
                sample_df = df.sample(n=sample_size, random_state=42)
                print(f"Fallback to random sampling: {len(sample_df)} rows")
                
        elif strategy == 'top_performers':
            # Sample top performers by overall rating
            if 'overall' in df.columns:
                sample_df = df.nlargest(sample_size, 'overall')
                print(f"Top performers sampling: {len(sample_df)} rows")
            else:
                sample_df = df.head(sample_size)
                print(f"Head sampling: {len(sample_df)} rows")
        
        else:
            sample_df = df.head(sample_size)
            print(f"Head sampling: {len(sample_df)} rows")
        
        # Show sample characteristics
        if 'overall' in sample_df.columns:
            print(f"Overall range: {sample_df['overall'].min()}-{sample_df['overall'].max()}")
            print(f"Average overall: {sample_df['overall'].mean():.1f}")
        
        return sample_df
    
    def implement_lazy_loading(self, file_path: str, chunk_size: int = 10000) -> None:
        """Demonstrate lazy loading with pandas chunks"""
        print(f"Lazy Loading Demo (chunk_size={chunk_size})")
        print("=" * 45)
        
        try:
            chunk_reader = pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)
            
            chunk_count = 0
            total_rows = 0
            memory_usage = []
            
            start_time = time.time()
            
            for chunk in chunk_reader:
                chunk_count += 1
                total_rows += len(chunk)
                current_memory = self.get_memory_usage()
                memory_usage.append(current_memory)
                
                # Process chunk (example: calculate statistics)
                if 'overall' in chunk.columns:
                    chunk_avg = chunk['overall'].mean()
                    print(f"Chunk {chunk_count}: {len(chunk)} rows, avg overall: {chunk_avg:.1f}")
                
                # Memory management
                if chunk_count % 10 == 0:
                    gc.collect()  # Force garbage collection
                    print(f"Memory cleanup at chunk {chunk_count}")
                
                # Stop after a few chunks for demo
                if chunk_count >= 5:
                    print(f" Stopping demo after {chunk_count} chunks")
                    break
            
            end_time = time.time()
            processing_time = end_time - start_time
            
            print(f"\nLazy Loading Results:")
            print(f"   Chunks processed: {chunk_count}")
            print(f"   Total rows: {total_rows:,}")
            print(f"   Processing time: {processing_time:.2f} seconds")
            print(f"   Average memory: {np.mean(memory_usage):.1f} MB")
            print(f"   Peak memory: {max(memory_usage):.1f} MB")
            
        except Exception as e:
            print(f"Error during lazy loading: {e}")
    
    def benchmark_operations(self, df: pd.DataFrame) -> Dict:
        """Benchmark common operations on the dataset"""
        print("Performance Benchmarking")
        print("=" * 30)
        
        benchmarks = {}
        
        operations = [
            ('Data Loading', lambda: df.copy()),
            ('Basic Statistics', lambda: df.describe()),
            ('Filtering', lambda: df[df['overall'] > 80] if 'overall' in df.columns else df.head()),
            ('Sorting', lambda: df.sort_values('overall') if 'overall' in df.columns else df),
            ('Grouping', lambda: df.groupby('player_positions')['overall'].mean() if all(col in df.columns for col in ['player_positions', 'overall']) else None)
        ]
        
        for operation_name, operation_func in operations:
            try:
                start_memory = self.get_memory_usage()
                start_time = time.time()
                
                result = operation_func()
                
                end_time = time.time()
                end_memory = self.get_memory_usage()
                
                execution_time = end_time - start_time
                memory_delta = end_memory - start_memory
                
                benchmarks[operation_name] = {
                    'time': execution_time,
                    'memory_delta': memory_delta,
                    'result_size': len(result) if hasattr(result, '__len__') else 'N/A'
                }
                
                print(f" {operation_name}: {execution_time:.3f}s, Memory: {memory_delta:+.1f}MB")
                
            except Exception as e:
                print(f"{operation_name}: Error - {str(e)}")
                benchmarks[operation_name] = {'error': str(e)}
        
        return benchmarks
    
    def get_optimization_recommendations(self, df: pd.DataFrame) -> List[str]:
        """Generate performance optimization recommendations"""
        recommendations = []
        
        # Check dataset size
        dataset_size_mb = df.memory_usage(deep=True).sum() / 1024**2
        row_count = len(df)
        
        if dataset_size_mb > 100:
            recommendations.append(f"Large dataset ({dataset_size_mb:.1f}MB) - Consider sampling for development")
        
        if row_count > 100000:
            recommendations.append(f"Many rows ({row_count:,}) - Implement pagination in UI")
        
        # Check data types
        memory_inefficient_cols = df.select_dtypes(include=['int64', 'float64']).columns
        if len(memory_inefficient_cols) > 5:
            recommendations.append("Optimize data types with downcast methods")
        
        # Check for categorical data
        object_cols = df.select_dtypes(include=['object']).columns
        for col in object_cols:
            if df[col].nunique() / len(df) < 0.5:
                recommendations.append(f"Convert '{col}' to categorical data type")
        
        # Check for missing values
        missing_data = df.isnull().sum().sum()
        if missing_data > len(df) * 0.1:
            recommendations.append("High missing data - implement efficient handling")
        
        return recommendations

# Example usage with mock data if FIFA data not available
print("Performance Optimization Demo")
print("=" * 35)

# Create a mock dataset for demonstration
np.random.seed(42)
mock_data = {
    'player_id': range(50000),
    'overall': np.random.randint(50, 100, 50000),
    'potential': np.random.randint(50, 100, 50000),
    'age': np.random.randint(16, 40, 50000),
    'value_eur': np.random.exponential(1000000, 50000),
    'player_positions': np.random.choice(['ST', 'CM', 'CB', 'GK', 'LW'], 50000),
    'league_name': np.random.choice(['Premier League', 'La Liga', 'Serie A', 'Bundesliga'], 50000)
}

mock_df = pd.DataFrame(mock_data)
print(f"Created mock dataset: {len(mock_df):,} rows")

# Initialize optimizer
optimizer = PerformanceOptimizer()

# Run optimizations
print("\n" + "="*50)
optimized_df = optimizer.optimize_dataframe_dtypes(mock_df)

print("\n" + "="*50)
sample_df = optimizer.create_sample_dataset(optimized_df, sample_size=5000)

print("\n" + "="*50)
benchmark_results = optimizer.benchmark_operations(sample_df)

print("\n" + "="*50)
recommendations = optimizer.get_optimization_recommendations(mock_df)
print("Optimization Recommendations:")
for rec in recommendations:
    print(f"   {rec}")

print("\nPerformance optimization demo completed!")

## 7. Caching

The apps use `@st.cache_data` so the CSVs are only read once per session.
The cell below is a small file-based imitation of that decorator to show how
cache hits and misses behave. It writes to `.cache/`, which is ignored by git.

In [ ]:
# Note: This demonstrates caching patterns used in the FIFA Dashboard
# In an actual Streamlit app, these would be used with @st.cache_data decorator

import hashlib
import pickle
import time
from functools import wraps
from typing import Any, Callable, Dict, List, Optional
import os

class CacheManager:
    """Simulate Streamlit's caching behavior for demonstration"""
    
    def __init__(self, cache_dir: str = ".cache"):
        self.cache_dir = cache_dir
        self.cache_stats = {'hits': 0, 'misses': 0}
        os.makedirs(cache_dir, exist_ok=True)
    
    def _get_cache_key(self, func_name: str, args: tuple, kwargs: dict) -> str:
        """Generate a unique cache key for function call"""
        key_data = f"{func_name}_{str(args)}_{str(sorted(kwargs.items()))}"
        return hashlib.md5(key_data.encode()).hexdigest()
    
    def _get_cache_path(self, cache_key: str) -> str:
        """Get file path for cached result"""
        return os.path.join(self.cache_dir, f"{cache_key}.cache")
    
    def cache_data(self, ttl: Optional[int] = None, show_spinner: bool = True):
        """Decorator to cache function results (simulates @st.cache_data)"""
        def decorator(func: Callable) -> Callable:
            @wraps(func)
            def wrapper(*args, **kwargs):
                # Generate cache key
                cache_key = self._get_cache_key(func.__name__, args, kwargs)
                cache_path = self._get_cache_path(cache_key)
                
                # Check if cached result exists and is valid
                if os.path.exists(cache_path):
                    if ttl is None or (time.time() - os.path.getmtime(cache_path)) < ttl:
                        try:
                            with open(cache_path, 'rb') as f:
                                result = pickle.load(f)
                            self.cache_stats['hits'] += 1
                            print(f"Cache hit for {func.__name__}")
                            return result
                        except Exception as e:
                            print(f" Cache read error: {e}")
                
                # Cache miss - execute function
                self.cache_stats['misses'] += 1
                print(f"Cache miss for {func.__name__} - executing...")
                
                if show_spinner:
                    print(f"Running {func.__name__}...")
                
                start_time = time.time()
                result = func(*args, **kwargs)
                execution_time = time.time() - start_time
                
                print(f"{func.__name__} completed in {execution_time:.2f}s")
                
                # Save result to cache
                try:
                    with open(cache_path, 'wb') as f:
                        pickle.dump(result, f)
                    print(f"Cached result for {func.__name__}")
                except Exception as e:
                    print(f" Cache write error: {e}")
                
                return result
            
            return wrapper
        return decorator
    
    def clear_cache(self, pattern: Optional[str] = None):
        """Clear cached results"""
        cleared_count = 0
        
        for filename in os.listdir(self.cache_dir):
            if filename.endswith('.cache'):
                if pattern is None or pattern in filename:
                    file_path = os.path.join(self.cache_dir, filename)
                    try:
                        os.remove(file_path)
                        cleared_count += 1
                    except Exception as e:
                        print(f"Error removing {filename}: {e}")
        
        print(f"Cleared {cleared_count} cache files")
        self.cache_stats = {'hits': 0, 'misses': 0}
    
    def get_cache_stats(self) -> Dict[str, Any]:
        """Get cache performance statistics"""
        total_requests = self.cache_stats['hits'] + self.cache_stats['misses']
        hit_rate = (self.cache_stats['hits'] / total_requests * 100) if total_requests > 0 else 0
        
        cache_files = [f for f in os.listdir(self.cache_dir) if f.endswith('.cache')]
        total_cache_size = sum(os.path.getsize(os.path.join(self.cache_dir, f)) for f in cache_files)
        
        return {
            'hits': self.cache_stats['hits'],
            'misses': self.cache_stats['misses'],
            'hit_rate': hit_rate,
            'cache_files': len(cache_files),
            'total_size_mb': total_cache_size / 1024**2
        }

# Initialize cache manager
cache_manager = CacheManager()

# Example cached functions for FIFA Dashboard

@cache_manager.cache_data(ttl=3600)  # Cache for 1 hour
def load_fifa_data_cached(file_path: str) -> pd.DataFrame:
    """Cached data loading function"""
    print(f"Loading data from {file_path}...")
    time.sleep(2)  # Simulate loading time
    
    # In real implementation, this would load actual FIFA data
    np.random.seed(42)
    data = {
        'player_id': range(10000),
        'long_name': [f'Player_{i}' for i in range(10000)],
        'overall': np.random.randint(50, 100, 10000),
        'potential': np.random.randint(50, 100, 10000),
        'value_eur': np.random.exponential(1000000, 10000)
    }
    
    return pd.DataFrame(data)

@cache_manager.cache_data(ttl=1800)  # Cache for 30 minutes
def calculate_player_statistics(df: pd.DataFrame) -> Dict[str, float]:
    """Cached statistics calculation"""
    print("Calculating player statistics...")
    time.sleep(1)  # Simulate processing time
    
    stats = {
        'total_players': len(df),
        'avg_overall': df['overall'].mean(),
        'avg_potential': df['potential'].mean(),
        'avg_value': df['value_eur'].mean(),
        'max_overall': df['overall'].max(),
        'min_overall': df['overall'].min()
    }
    
    return stats

@cache_manager.cache_data(ttl=7200)  # Cache for 2 hours
def train_value_prediction_model(df: pd.DataFrame) -> Dict[str, Any]:
    """Cached ML model training"""
    print("Training value prediction model...")
    time.sleep(3)  # Simulate training time
    
    # In real implementation, this would train an actual ML model
    model_info = {
        'model_type': 'RandomForestRegressor',
        'features': ['overall', 'potential'],
        'accuracy': None,  # placeholder, no real model is trained here
        'training_samples': len(df),
        'trained_at': time.time()
    }
    
    return model_info

@cache_manager.cache_data(ttl=600)  # Cache for 10 minutes
def filter_players_by_criteria(df: pd.DataFrame, min_overall: int = 80, 
                              max_age: int = 30) -> pd.DataFrame:
    """Cached filtering operation"""
    print(f"Filtering players (overall >= {min_overall}, age <= {max_age})...")
    time.sleep(0.5)  # Simulate processing time
    
    # Add age column for filtering demo
    df['age'] = np.random.randint(16, 40, len(df))
    
    filtered_df = df[
        (df['overall'] >= min_overall) & 
        (df['age'] <= max_age)
    ]
    
    return filtered_df

# Demonstrate caching behavior
print("Caching Demonstration")
print("=" * 25)

# First calls - cache misses
print("\nFirst execution (cache misses):")
print("-" * 35)

data_df = load_fifa_data_cached("data/players.csv")
stats = calculate_player_statistics(data_df)
model = train_value_prediction_model(data_df)
filtered = filter_players_by_criteria(data_df, min_overall=85)

print(f"\nStatistics: {stats}")
print(f"Model info: {model['model_type']} (placeholder)")
print(f"Filtered players: {len(filtered)}")

# Second calls - cache hits
print("\nSecond execution (cache hits):")
print("-" * 35)

data_df = load_fifa_data_cached("data/players.csv")
stats = calculate_player_statistics(data_df)
model = train_value_prediction_model(data_df)
filtered = filter_players_by_criteria(data_df, min_overall=85)

# Show cache statistics
print("\nCache Performance:")
print("-" * 20)
cache_stats = cache_manager.get_cache_stats()
for key, value in cache_stats.items():
    if key == 'hit_rate':
        print(f"   {key}: {value:.1f}%")
    elif key == 'total_size_mb':
        print(f"   {key}: {value:.2f} MB")
    else:
        print(f"   {key}: {value}")

# Cache management examples
print("\nCache Management:")
print("-" * 20)

# Show cache optimization recommendations
def get_cache_recommendations(stats: Dict) -> List[str]:
    recommendations = []
    
    if stats['hit_rate'] < 50:
        recommendations.append("Low hit rate - consider longer TTL values")
    
    if stats['total_size_mb'] > 100:
        recommendations.append("Large cache size - consider periodic cleanup")
    
    if stats['cache_files'] > 50:
        recommendations.append(" Many cache files - implement cache rotation")
    
    return recommendations

recommendations = get_cache_recommendations(cache_stats)
if recommendations:
    print("Cache Optimization Recommendations:")
    for rec in recommendations:
        print(f"   {rec}")
else:
    print("No cache issues found")

print("\nCaching Best Practices for FIFA Dashboard:")
print("   1. Cache data loading operations (@st.cache_data)")
print("   2. Use appropriate TTL values (1-24 hours for data)")
print("   3. Cache ML model training (longer TTL)")
print("   4. Cache expensive calculations and aggregations")
print("   5. Clear cache when underlying data changes")
print("   6. Monitor cache hit rates and sizes")
print("   7. Use hash_funcs for complex objects if needed")

## 8. Troubleshooting

**Missing packages**

```bash
pip install -r requirements.txt
```

**File not found** - the apps use relative paths, so start them from the repository root:

```python
import os
print(os.getcwd())
print(os.listdir("data/"))
```

**Memory** - sample the data or downcast types:

```python
df_sample = df.sample(n=10000)
df = df.astype({'player_id': 'uint32', 'overall': 'uint8'})
for chunk in pd.read_csv("data/male_players.csv", chunksize=10000):
    ...
```

**Port already in use**

```bash
streamlit run app.py --server.port 8502
netstat -ano | findstr :8501    # Windows
lsof -ti:8501 | xargs kill      # macOS/Linux
```

## Quick checks

```python
def test_data_integrity(df):
    tests = []
    required_cols = ['player_id', 'long_name', 'overall']
    missing_cols = [col for col in required_cols if col not in df.columns]
    tests.append(("Required columns", len(missing_cols) == 0, missing_cols))

    duplicates = df['player_id'].duplicated().sum()
    tests.append(("No duplicates", duplicates == 0, f"{duplicates} duplicates"))

    valid_overall = df['overall'].between(0, 100).all()
    tests.append(("Overall range 0-100", valid_overall, "Invalid overall values"))
    return tests
```

If the CSV holds several FIFA versions (`fifa_version` column), the same
`player_id` appears more than once, so this check will flag duplicates.

## Deployment notes

A minimal Dockerfile would look like this (not included in the repo):

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.address", "0.0.0.0"]
```

The data files are not in the repository, so they have to be copied or mounted
into the container separately. For Streamlit Community Cloud the same applies:
the CSVs need to be available to the app at `data/`.